In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Navier-Stokes Solver with NVIDIA Warp

## Overview

Having covered Warp fundamentals in the previous notebook, we will now build a 2-D fluid solver entirely in Python using Warp. The solver serves as a vehicle for learning the framework so you can apply it in your own domain.

Along the way, you will learn:

* **Memory management**: mapping numerical algorithm variables to GPU arrays, deciding what to allocate, and using `wp.zeros()` and `wp.array()` to initialize them
* **Warp kernels**: writing Warp kernels for computational physics operations
* **Kernel orchestration**: launching kernels with `wp.launch()` and `wp.launch_tiled()`, and capturing repetitive kernel launches as CUDA graphs with `wp.ScopedCapture()` to improve performance

**By the end of this lab**, we will have built a GPU-accelerated physics solver while gaining hands-on experience applying Warp's features to computational physics problems.

**In the next notebook**, we will walk through the steps needed to make this solver differentiable using Warp's autodiff capabilities.

<img src="https://media.githubusercontent.com/media/NVIDIA/accelerated-computing-hub/main/tutorials/warp/notebooks/images/navier-stokes/turbulence_256x256_Re1000.gif" alt="Turbulence simulation preview" width="400">

In [ ]:
import os
import subprocess
import sys

# Fetch local helpers and assets when this notebook runs in Google Colab.
if os.getenv("COLAB_RELEASE_TAG"):
    repo_dir = "/content/accelerated-computing-hub"
    repo_url = os.getenv(
        "ACH_REPO_URL",
        "https://github.com/NVIDIA/accelerated-computing-hub.git",
    )
    repo_ref = os.getenv("ACH_REPO_REF", "main")

    if not os.path.exists(repo_dir):
        print("Cloning Accelerated Computing Hub.")
        subprocess.run(
            ["git", "clone", "--depth=1", "--branch", repo_ref, repo_url, repo_dir],
            check=True,
        )

    os.chdir(os.path.join(repo_dir, "tutorials", "warp", "notebooks"))
    print(f"Working directory: {os.getcwd()}")

    print("Installing PIP packages.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "warp-lang", "numpy", "matplotlib", "pillow"],
        check=True,
        stdout=subprocess.DEVNULL,
    )
    print("Colab setup complete.")


In [ ]:
from importlib.metadata import PackageNotFoundError, version

packages = [
    "warp-lang",
    "numpy",
    "matplotlib",
    "pillow",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

# Shorten exception tracebacks for readability in this notebook
%xmode Minimal

In [ ]:
import numpy as np
import warp as wp

if wp.get_cuda_device_count() > 0:
    print("GPU detected successfully")
else:
    raise RuntimeError("No CUDA-capable GPU detected. This notebook assumes at least one NVIDIA GPU.")

---
## Problem Statement

We are building a solver for 2-D decaying turbulence governed by the **Navier–Stokes equations** (*equations that we solve*) in the **vorticity–stream function** (*variables that we solve for*) formulation. We solve it on a **uniform structured grid using finite-difference schemes** (*type of kernels that we will need to write*).

<p align="center">
  <img src="./images/navier-stokes/structured_grid_stencil.svg" alt="Vorticity field with grid overlay" width="40%">
</p>

**Inputs:**
- Initial vorticity field $\omega_0$ on an $N \times N$ uniform grid
- Reynolds number $\text{Re}$
- Grid resolution $N$

**Outputs:**
- Vorticity $\omega(t)$ at each timestep

### Equations, variables and methods

The **first equation** that we will work through is the vorticity-transport equation given below:

$$\frac{\partial \omega}{\partial t} + \frac{\partial \omega}{\partial x}\frac{\partial \psi}{\partial y} - \frac{\partial \omega}{\partial y}\frac{\partial \psi}{\partial x} = \frac{1}{\text{Re}}\left(\frac{\partial^2 \omega}{\partial x^2} + \frac{\partial^2 \omega}{\partial y^2}\right). \tag{1}$$

**Key takeaway**: To compute the spatial derivatives at any grid point $(i,j)$, we only need to access the field values at **its surrounding four neighbors** and **perform elementary mathematical operations** (addition, division, multiplication). We do this computation for many points ($N \times N$), making it well suited for a GPU kernel!

Likewise, we approximate the $\partial \omega/\partial t$ term as follows, where superscript $n$ denotes a given timestep:

$$\frac{\partial \omega}{\partial t} \approx \frac{\omega^{(n+1)}_{i,j} - \omega^{(n)}_{i,j}}{\Delta t}.$$

Similar to the spatial-derivative computation above, this time update also needs to happen at all $N \times N$ grid points.

The **second equation** that we need to solve is a Poisson equation linking $\omega$ and $\psi$,

$$\frac{\partial^2 \psi}{\partial x^2} + \frac{\partial^2 \psi}{\partial y^2} = -\omega. \tag{2}$$

The boundary conditions that we use here enable us to Fourier transform the **PDE** above into a simple **algebraic pointwise division** for each $[k_x, k_y]$ pair.

$$ \hat{\psi}(k_x, k_y) = \frac{\hat{\omega}(k_x, k_y)}{k_x^2 + k_y^2} = \frac{\hat{\omega}(k_x, k_y)}{||k||^2}. \tag{3} $$

**Key takeaway**: This transformation allows us to leverage Warp's tile-based APIs for FFT, `wp.tile_fft()` and `wp.tile_ifft()`, to solve Eq. (2) above.


### Solution Approach

For the two equations, we have two stages that we map onto the Warp programming model.

1. **Vorticity update**: approximate the diffusion and advection terms using neighboring grid values, then update $\omega$ with the forward Euler step.
2. **Poisson solve**: Given the new $\omega$, solve Eq. (2) for $\psi$ using Warp's tile-based APIs (Eq. 3).

<p align="center">
  <img src="./images/navier-stokes/navier_stokes_timestep.svg" alt="Solver timestep loop" width="600">
</p>


---
## Warp Programming Model

Our Warp-based solver follows a three-step pattern:

1. **Memory**: Allocate arrays **directly on the GPU** or **interoperate with NumPy** (`wp.zeros()`, `wp.array()`, `wp.copy()`, etc.)
2. **Kernels**: Define our **compute-intensive operations** as `@wp.kernel()` functions that run on the GPU
3. **Orchestration**: Launch those kernels on the GPU to perform the computation (`wp.launch()`, `wp.launch_tiled()`)

<p align="center">
  <img src="./images/navier-stokes/warp_runtime_overview.svg" alt="Warp runtime overview" width="900">
</p>

---
## Initialization 

### Simulation Parameters

We solve on a square domain of size $2\pi$, discretized on a $512 \times 512$ grid.

In [ ]:
# Grid resolution
N_GRID = 512

# Box size
LEN = 2 * np.pi

# Delta t for timestepping (by how much time does a simulation advance per iteration)
DT = 0.0005

# Reynolds number (global simulation parameter)
RE = 1000.0

# Derived grid constants (captured by Warp kernels)
H = LEN / N_GRID  # grid size

### Exercise 1: Array Allocation

<p align="center">
  <img src="./images/navier-stokes/navier_stokes_timestep.svg" alt="Solver timestep loop" width="600">
</p>

$$\frac{\boxed{\omega^{(n+1)}_{i,j}} - \boxed{\omega^{(n)}_{i,j}}}{\Delta t} = \frac{1}{\text{Re}} \left(\frac{\partial^2}{\partial x^2} + \frac{\partial^2}{\partial y^2}\right) {\omega^{(n)}_{i,j}} - J(\boxed{\psi_{i, j}^{(n)}}, \omega_{i,j}^{(n)}).$$

Here, the superscript $n$ denotes the timestep index and the subscripts $i, j$ denote the grid point coordinates.

Each boxed term is a distinct 2-D field, so it needs its own GPU array allocation.

| Array | Shape | Dtype | Purpose |
|-------|-------|-------|---------|
| `omega_0` | `(N_GRID, N_GRID)` | `float` | Vorticity at timestep $n$ |
| `omega_1` | `(N_GRID, N_GRID)` | `float` | Vorticity at timestep $n+1$ |
| `psi` | `(N_GRID, N_GRID)` | `float` | Stream function at any time |

**Note**: We need two vorticity arrays, `omega_0` and `omega_1`, because the time update requires both the current state and the newly computed state. By contrast, `psi` does not need to be stored at both time levels: once $\omega$ has been updated, we solve for the corresponding $\psi$ and overwrite the same `psi` array at every timestep.

In [ ]:
# ============================================================
# EXERCISE 1: Fill in the wp.zeros / wp.array calls below
# ============================================================

# Vorticity at timestep n
omega_0 = None # TODO, Replace None with Warp based GPU array allocation

# Vorticity at timestep n+1
omega_1 = wp.zeros_like(omega_0)

# Stream function
psi = None # TODO, Replace None with Warp based GPU array allocation

### Validate array allocations

In [ ]:
for name, arr in [("omega_0", omega_0), ("omega_1", omega_1), ("psi", psi)]:
    assert arr is not None, f"{name} is still None - fill in the allocation"
    assert isinstance(arr, wp.array), f"{name} should be a wp.array, got {type(arr)}"
    assert arr.shape == (N_GRID, N_GRID), f"{name}.shape = {arr.shape}, expected ({N_GRID}, {N_GRID})"
    assert arr.dtype == wp.float32, f"{name}.dtype = {arr.dtype}, expected float"

print("All arrays allocated correctly")

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
omega_0 = wp.zeros((N_GRID, N_GRID), dtype=float)
omega_1 = wp.zeros_like(omega_0)
psi = wp.zeros((N_GRID, N_GRID), dtype=float)
```
</details>

---
## The `advance_vorticity_by_dt()` function

<p align="center">
  <img src="./images/navier-stokes/time_marching.svg" alt="Evolving the vorticity field" width="540">
</p>

We encapsulate one timestep of the solver, as shown above, into a single `advance_vorticity_by_dt()` function that orchestrates different **Warp arrays** and **Warp kernel launches** together.  

Within the `advance_vorticity_by_dt()` function, we have two building blocks that we need for the two equations:

- **Building Block 1**: Vorticity update (finite-difference stencils + Forward Euler step)
- **Building Block 2**: Poisson solve for $\psi$ from $\omega$ via FFT

<p align="center">
  <img src="./images/navier-stokes/solver_pipeline_overview.svg" alt="Solver pipeline highlighting discretization step" width="800">
</p>

```python
def advance_vorticity_by_dt(omega_0, omega_1, psi):

    # === Building Block 1: Solve equation 1 to update vorticity ===
    # Compute finite difference approximations of Eq. 1, advance omega by one timestep
    
    # === Building Block 2: Solve equation 2 using FFT ===
    # 2-D FFT to obtain vorticity in Fourier space
    # Solve the Poisson equation in Fourier space
    # 2-D inverse FFT to obtain stream function in physical space
```

---
## Building Block 1 - Vorticity Update

<p align="center">
  <img src="./images/navier-stokes/solver_pipeline_simple_discretization.svg" alt="Solver pipeline highlighting discretization step" width="800">
</p>

In this section, you will build the first kernel as an exercise. We will then walk through the second kernel and, in the end, see how kernel fusion lets us encapsulate everything in a single Warp kernel.

1. **`diffusion_kernel`**: 5-point Laplacian stencil to calculate $\nabla^2 \omega$ (Exercise 2)
2. **`advection_kernel`**: Jacobian $J(\psi, \omega)$
3. **`viscous_advection_kernel`**: Fused kernel combining both + Euler step (walkthrough)

### Exercise 2: `diffusion_kernel`

If you remember, one of the terms that we wanted to approximate was the diffusion term below:

<p align="center">
  <img src="https://media.githubusercontent.com/media/NVIDIA/accelerated-computing-hub/main/tutorials/warp/notebooks/images/navier-stokes/diffusion.png" alt="Five-point stencil for diffusion" width="700">
</p>

**Key Takeaway**: Let's say you launch a 2-D grid of threads and assign each $(i, j)$ thread the responsibility for the calculation above, using ``omega`` for the $(i, j)$ cell and writing to an output Warp array named ``laplacian``. What would your Warp kernel look like?

In [ ]:
@wp.func
def cyclic_index(idx: int, n: int) -> int:
    """Map any index to [0, n-1] for periodic boundary conditions."""
    ret_idx = idx % n
    if ret_idx < 0:
        ret_idx += n
    return ret_idx

# ============================================================
# EXERCISE 2: Calculate the Laplacian of omega using the 5-point stencil
# ============================================================

@wp.kernel
def diffusion_kernel(
    omega: wp.array2d[float],
    laplacian: wp.array2d[float],
):
    """Compute the Laplacian of omega using the 5-point stencil.

    Args:
        omega: Input vorticity field on N_GRID x N_GRID.
        laplacian: Output array for the computed Laplacian.
    """
    i, j = wp.tid()

    # Periodic neighbor indices, wraps around the grid edges
    i_left = cyclic_index(i - 1, N_GRID)
    i_right = cyclic_index(i + 1, N_GRID)
    j_down = cyclic_index(j - 1, N_GRID)
    j_top = cyclic_index(j + 1, N_GRID)

    # left neighbor: [i_left, j], right neighbor: [i_right, j], top neighbor: [i, j_top], bottom neighbor: [i, j_down]
    laplacian[i, j] = None  # TODO

#### Validate diffusion kernel

We test against an analytical case where the Laplacian is known exactly:

$$\omega = \sin(k_x x)\sin(k_y y) \quad\Rightarrow\quad \nabla^2\omega = -(k_x^2 + k_y^2)\sin(k_x x)\sin(k_y y)$$

The cell below compares the kernel output against this analytical solution with $k_x=2$, $k_y=3$.

In [ ]:
from helpers.navier_stokes.utils import validate_diffusion

validate_diffusion(diffusion_kernel, n_grid=512, kx=2, ky=3)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
    laplacian[i, j] = (omega[i_left, j] + omega[i_right, j] + omega[i, j_top] + omega[i, j_down] - 4.0 * omega[i, j]) / (H * H)
```
</details>

### Walkthrough: `advection_kernel`

The thought process from the exercise above carries over directly to the other term that we want to approximate. **To reiterate, each thread $(i, j)$ is responsible for the computation at the corresponding grid cell. That thread loads the required information from global memory, performs the calculation, and writes the result back to global memory**. Let's walk through the example to reinforce that idea.

<table align="center"><tr>
<td><img src="https://media.githubusercontent.com/media/NVIDIA/accelerated-computing-hub/main/tutorials/warp/notebooks/images/navier-stokes/advection.png" alt="Omega stencil" width="700"></td>
</tr></table>

**Note**: The main difference here is that you now need two input arrays, ``omega`` and ``psi``, to perform the computation. You then write the result to an ``advection`` Warp array.

In [ ]:
@wp.kernel
def advection_kernel(
    omega: wp.array2d[float],
    psi: wp.array2d[float],
    advection_out: wp.array2d[float],
):
    """Compute the advection term J(psi, omega) at each grid point.

    Args:
        omega: Input vorticity field.
        psi: Input stream function field.
        advection_out: Output array for the Jacobian.
    """
    i, j = wp.tid()

    # Periodic neighbor indices, wraps around the grid edges
    i_left = cyclic_index(i - 1, N_GRID)
    i_right = cyclic_index(i + 1, N_GRID)
    j_down = cyclic_index(j - 1, N_GRID)
    j_top = cyclic_index(j + 1, N_GRID)

    # left neighbor: [i_left, j], right neighbor: [i_right, j], top neighbor: [i, j_top], bottom neighbor: [i, j_down]
    advection_out[i, j] = (
        (omega[i_right, j] - omega[i_left, j]) / (2.0 * H) * (psi[i, j_top] - psi[i, j_down]) / (2.0 * H)
        - (omega[i, j_top] - omega[i, j_down]) / (2.0 * H) * (psi[i_right, j] - psi[i_left, j]) / (2.0 * H)
    )

#### Validate advection kernel

We test with functions that yield an analytically known Jacobian:

$$\psi = \sin(k_x x)\sin(k_y y), \quad \omega = \cos(k_x x)\cos(k_y y) \quad\Rightarrow\quad J(\psi,\omega) = \frac{k_x k_y}{2}\bigl(\cos(2k_x x) - \cos(2k_y y)\bigr).$$

The cell below compares the kernel output against this analytical solution with $k_x=2$, $k_y=3$.

In [ ]:
from helpers.navier_stokes.utils import validate_advection

validate_advection(advection_kernel, n_grid=512, kx=2, ky=3)

### Walkthrough: Fused Kernel

Instead of launching diffusion and advection as separate kernels, `viscous_advection_kernel()` **fuses both calculations with the forward Euler time step**. 

<p align="center">
  <img src="./images/navier-stokes/viscous_advection_fusion.svg" alt="Unfused vs fused kernel launches" width="800">
</p>

<p align="center">
  <img src="https://media.githubusercontent.com/media/NVIDIA/accelerated-computing-hub/main/tutorials/warp/notebooks/images/navier-stokes/simt-update-schematic.png" alt="5-point stencil for advection and diffusion" width="700">
</p>

**Key takeaway**: The idea that enables us to do kernel fusion here is that **the calculations of the spatial derivatives ($\partial/\partial x$, $\partial/\partial y$) are all done on $\omega^{(n)}$** and then the update is applied to $\omega^{(n+1)}$. If your problem is amenable to it, always try to fuse kernels for different operations together!

This is the kernel launched in Building Block 1 of `advance_vorticity_by_dt()`. 

In [ ]:
@wp.kernel
def viscous_advection_kernel(
    omega_0: wp.array2d[float],
    psi: wp.array2d[float],
    omega_1: wp.array2d[float],
):
    """Compute advection + diffusion and advance vorticity using forward Euler.

    Fuses the diffusion and advection stencil computations with the Euler
    time step in a single kernel launch.

    Args:
        omega_0: Input vorticity field at the beginning of the timestep.
        psi: Input stream function field at the beginning of the timestep.
        omega_1: Output vorticity field at the end of the timestep.
    """
    i, j = wp.tid()

    # Periodic neighbor indices, wraps around the grid edges
    i_left = cyclic_index(i - 1, N_GRID)
    i_right = cyclic_index(i + 1, N_GRID)
    j_top = cyclic_index(j + 1, N_GRID)
    j_down = cyclic_index(j - 1, N_GRID)

    # Diffusion
    diff = (omega_0[i_left, j] + omega_0[i_right, j] +
            omega_0[i, j_top] + omega_0[i, j_down] -
            4.0 * omega_0[i, j]) / (H * H)

    # Advection
    adv = (
        (omega_0[i_right, j] - omega_0[i_left, j]) / (2.0 * H) * (psi[i, j_top] - psi[i, j_down]) / (2.0 * H)
        - (omega_0[i, j_top] - omega_0[i, j_down]) / (2.0 * H) * (psi[i_right, j] - psi[i_left, j]) / (2.0 * H)
    )

    # Forward Euler: omega^(n+1) = omega^n + dt * R(omega^n)
    omega_1[i, j] = omega_0[i, j] + DT * (diff / RE - adv)

---
## Building Block 2 - Tile-Based Fast Fourier Transform (FFT) Kernels

<p align="center">
  <img src="./images/navier-stokes/solver_pipeline_simple_poisson.svg" alt="Solver pipeline highlighting Poisson step" width="800">
</p>

```python
def advance_vorticity_by_dt(omega_0, omega_1, psi):

    # === Building Block 1: Solve equation 1 to update vorticity ===
    # Compute finite difference approximations of Eq. 1, advance omega by one timestep
    
    # === Building Block 2: Solve equation 2 using FFT ===
    # 2-D FFT to obtain vorticity in Fourier space
    # Solve the Poisson equation in Fourier space
    # 2-D inverse FFT to obtain stream function in physical space
```

Now, we turn our attention to the second block of the solver, solving the Poisson equation using FFT.

$$ \hat{\psi}(k_x, k_y) = \frac{\hat{\omega}(k_x, k_y)}{k_x^2 + k_y^2} = \frac{\hat{\omega}(k_x, k_y)}{||k||^2}. \tag{3} $$

<p align="center">
  <img src="./images/navier-stokes/poisson_pipeline.svg" alt="Fourier Poisson solver pipeline: omega to psi via FFT, spectral division, and IFFT">
</p>

### Warp's tile-FFT API

Since version 1.5.0, Warp supports tile-based programming. For efficient FFT computation, we want a block of threads cooperating on an entire row rather than each thread working independently on a single element. This is precisely what Warp's tile-based APIs enable. We will specifically make use of `wp.tile_fft()` and `wp.tile_ifft()` functions.

> **`wp.tile_fft(inout)`**
  >
  > Compute the forward FFT along the last dimension of an N-D tile of data. Leading dimensions are treated as independent batches.
  >
  > **Supported dtypes:** `vec2f`, `vec2d`
  >
  > **Supported FFT sizes by backend:**
  > - **CPU**: any power-of-two length, and non-power-of-two lengths up to 4096
  > - **GPU with libmathdx**: any FFT length
  > - **GPU fallback** without libmathdx, or with `enable_mathdx_fft=False`: power-of-two sizes only, and the FFT size must be divisible by `block_dim`
  >
  > *Source: [Warp documentation](https://nvidia.github.io/warp/stable/language_reference/builtins.html#warp.tile_fft)*

`wp.tile_fft()` transforms the last dimension of its input tile. In the kernels below, each tile contains one row of the 2-D complex array, so the last-dimension transform is a row FFT. To achieve a full 2-D FFT, we compose the row-FFT kernel with a transpose kernel. **How does this composition look in practice?**

We decompose the 2-D FFT (or inverse FFT) as follows:

<p align="center">
  <img src="./images/navier-stokes/fft_2d_pipeline.svg" alt="2-D FFT decomposition: row FFT, transpose, row FFT">
</p>

This means we need exactly two Warp kernels:
- **`fft_tiled`** (tile-based): 1-D FFT on each row using `wp.tile_fft()`
- **`transpose`** (SIMT): rearranges the 2-D array so columns become rows

**Key Takeaway** - At times, we might need to compose kernels and Warp API calls together to achieve the final mathematical operation that we want to perform!

### Type Conversion

Warp does not have a native complex number type. Instead, its FFT APIs use `wp.vec2f`, a 2-component vector of `float32`, to represent complex values, where `.x` is the real part and `.y` is the imaginary part. Our physical fields (`omega`, `psi`) are stored as plain `float` arrays, so we need two helper kernels: one to pack real values into `wp.vec2f` (setting the imaginary part to zero), and one to extract the real part back out after the inverse FFT.

In [ ]:
@wp.kernel
def copy_float_to_complex(
    omega: wp.array2d[float], omega_complex: wp.array2d[wp.vec2f]
):
    """Pack a real-valued field into a complex array (imaginary part set to zero).

    Args:
        omega: Real-valued input field.
        omega_complex: Output complex array (vec2f with .y = 0).
    """
    i, j = wp.tid()
    omega_complex[i, j] = wp.vec2f(omega[i, j], 0.0)

### FFT / IFFT using Warp's tile API

<p align="center">
  <img src="./images/navier-stokes/diagram8b_spatial_to_spectral.svg" alt="Tile FFT concept">
</p>

``fft_tiled()`` and ``ifft_tiled()`` are tile-based Warp kernels that apply a 1-D FFT or IFFT to each row of a 2-D complex array. Each kernel loads one row into a tile, transforms it in place with ``wp.tile_fft()`` or ``wp.tile_ifft()``, and stores the transformed row back. These kernels are then composed with ``transpose()`` to build the full 2-D FFT/IFFT.

**Launch parameters:** `wp.launch_tiled(fft_tiled, dim=[N_GRID, 1], block_dim=N_GRID//2)` means:
- `dim=[N_GRID, 1]`: N_GRID tiles, one for each row
- `block_dim=N_GRID//2`: each row tile is processed by N_GRID/2 cooperating threads. For the GPU fallback backend, the FFT size must be divisible by `block_dim`; the `N_GRID` used here is a power of two and satisfies that constraint.

**Note**: Warp provides several different [tile-based APIs](https://nvidia.github.io/warp/stable/user_guide/tiles.html) for efficient linear algebra operations. 

In [ ]:
@wp.kernel(module="dft_kernels")
def fft_tiled(x: wp.array2d[wp.vec2f], y: wp.array2d[wp.vec2f]):
    """Perform 1-D FFT on each row using wp.tile_fft().

    Args:
        x: Input 2-D complex array.
        y: Output 2-D complex array (FFT of each row).
    """
    i = wp.tid() # obtain block index
    a = wp.tile_load(x, shape=(1, N_GRID), offset=(i, 0)) # load one row of ``x`` per block
    wp.tile_fft(a) # FFT in-place
    wp.tile_store(y, a, offset=(i, 0)) # store result in one row of ``y``


@wp.kernel(module="dft_kernels")
def ifft_tiled(x: wp.array2d[wp.vec2f], y: wp.array2d[wp.vec2f]):
    """Perform 1-D inverse FFT on each row using wp.tile_ifft().

    Args:
        x: Input 2-D complex array.
        y: Output 2-D complex array (IFFT of each row).
    """
    i = wp.tid()
    a = wp.tile_load(x, shape=(1, N_GRID), offset=(i, 0))
    wp.tile_ifft(a)
    wp.tile_store(y, a, offset=(i, 0))

#### Validate FFT / IFFT (roundtrip test)

FFT a $\sin(x)$ signal, then IFFT back. The spectrum should peak at $k = \pm 1$ and the reconstructed signal should match the original.

In [ ]:
from helpers.navier_stokes.utils import validate_fft_roundtrip

fig, axes, max_error = validate_fft_roundtrip(
    fft_kernel=fft_tiled,
    ifft_kernel=ifft_tiled,
    n_grid=N_GRID,
    tile_m=1,
    tile_n=N_GRID,
    block_dim=N_GRID // 2,
)

### Transpose

The transpose kernel is the bridge between the two passes. Each thread copies one element from position `(i, j)` to `(j, i)`. The array type is `wp.vec2f` because we're transposing complex data mid-FFT.

#### Exercise 3: `transpose` kernel

One attempt at transposing a matrix on the GPU might be to do it **in-place**: every thread reads `x[j, i]` and writes it to `x[i, j]`. Let's try that and see what happens.

In [ ]:
@wp.kernel
def transpose_inplace(x: wp.array2d[wp.vec2f]):
    """Attempt an in-place transpose."""
    i, j = wp.tid()
    x[i, j] = x[j, i]

#### Validate transpose

Transpose an upper triangular matrix - it should become lower triangular:

$$
\begin{bmatrix} 1 & 1 & 1 & 1 \\ 0 & 1 & 1 & 1 \\ 0 & 0 & 1 & 1 \\ 0 & 0 & 0 & 1 \end{bmatrix}
\xrightarrow{\text{transpose}}
\begin{bmatrix} 1 & 0 & 0 & 0 \\ 1 & 1 & 0 & 0 \\ 1 & 1 & 1 & 0 \\ 1 & 1 & 1 & 1 \end{bmatrix}
$$

In [ ]:
from helpers.navier_stokes.utils import validate_transpose

validate_transpose(transpose_inplace, n_test=256)

The above results do not look correct. The problem is that an in-place transpose creates read/write hazards: some threads overwrite values before other threads have a chance to read them.

Let's fix that by writing the transpose into a separate output array.

In [ ]:
@wp.kernel
def transpose(x: wp.array2d[wp.vec2f], y: wp.array2d[wp.vec2f]):
   pass #TODO

In [ ]:
from helpers.navier_stokes.utils import validate_transpose

validate_transpose(transpose, n_test=256)

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
@wp.kernel
def transpose(x: wp.array2d[wp.vec2f], y: wp.array2d[wp.vec2f]):
    """Transpose a 2-D complex array: y[i, j] = x[j, i]."""
    i, j = wp.tid()
    y[i, j] = x[j, i]
```
</details>

### Fourier-Space Solve

The pointwise division in Fourier space is given by $\hat{\psi} = \hat{\omega} \cdot \|k\|^{-2}$. To implement this, we first precompute $1 / |k|^2$ on the grid, leaving the zero mode at zero to avoid division by zero. We then apply that factor elementwise to $\hat{\omega}$ to obtain $\hat{\psi}$.

In [ ]:
# Precompute 1/(kx^2 + ky^2) in NumPy for the spectral Poisson solver
# Note that inv_k_sq_np is a "NumPy" array, not a Warp array
k = np.fft.fftfreq(N_GRID, d=1.0 / N_GRID)
kx, ky = np.meshgrid(k, k)
k2 = kx**2 + ky**2
inv_k_sq_np = np.zeros_like(k2)
nonzero = k2 != 0
inv_k_sq_np[nonzero] = 1.0 / k2[nonzero]
inv_k_sq_np = inv_k_sq_np.astype(np.float32)

# Visualize the precomputed field
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 4))
plt.imshow(np.log10(inv_k_sq_np + 1e-12), cmap="viridis", origin="lower")
plt.colorbar(label=r"$\log_{10}(1/|k|^2)$")
plt.title(r"Precomputed $1/|k|^2$ (log scale)")
plt.tight_layout()
plt.show()

inv_k_sq = wp.array(inv_k_sq_np, dtype=float)

In [ ]:
@wp.kernel
def extract_real_and_scale(
    scale: float,
    complex_array: wp.array2d[wp.vec2f],
    real_array: wp.array2d[float],
):
    """Extract real part from a complex array and divide by a scale factor.

    Args:
        scale: Divisor applied to the real part (typically N^2 for FFT normalization).
        complex_array: Input complex array.
        real_array: Output real-valued array.
    """
    i, j = wp.tid()
    real_array[i, j] = complex_array[i, j].x / scale
    
@wp.kernel
def multiply_k2_inverse(
    inv_k_sq: wp.array2d[float],
    omega_hat: wp.array2d[wp.vec2f],
    psi_hat: wp.array2d[wp.vec2f],
):
    """Solve Poisson equation in Fourier space: psi_hat = omega_hat / |k|^2.

    Args:
        inv_k_sq: Precomputed 1/(kx^2 + ky^2) array.
        omega_hat: Vorticity in Fourier space.
        psi_hat: Output stream function in Fourier space.
    """
    i, j = wp.tid()
    psi_hat[i, j] = omega_hat[i, j] * inv_k_sq[i, j]

---
## Assembling `advance_vorticity_by_dt()`

**All the kernels we need to complete the loop are now built.**

- The `viscous_advection_kernel()` kernel helps us perform the first step in one go
- The Fourier Poisson solver was a bit more involved; it required building the `fft_tiled()`, `ifft_tiled()`, and `transpose()` kernels, along with a couple of other small helper kernels to solve that one equation.

**Key Takeaway**: Sometimes the operations that you want for a simulation fit well in a single kernel; at other times, you need to orchestrate multiple kernels together to achieve the desired task.

The `advance_vorticity_by_dt()` function is the main orchestrator of all the kernels that we have built. Let's go through it!

<p align="center">
  <img src="./images/navier-stokes/solver_pipeline_simple.svg" alt="Solver Pipeline" width="800">
</p>

```python
def advance_vorticity_by_dt(omega_0, omega_1, psi):

    # === FINISHED: Building Block 1: Solve equation 1 to update vorticity ===
    # Compute finite difference approximations of Eq. 1, advance omega by one timestep
    
    # === FINISHED: Building Block 2: Solve equation 2 using FFT ===
    # 2-D FFT to obtain vorticity in Fourier space
    # Solve the Poisson equation in Fourier space
    # 2-D inverse FFT to obtain stream function in physical space
```

In [ ]:
# Allocate temporary buffers for FFT operations (outside advance_vorticity_by_dt())
omega_complex = wp.zeros((N_GRID, N_GRID), dtype=wp.vec2f)
fft_temp_1 = wp.zeros((N_GRID, N_GRID), dtype=wp.vec2f)
fft_temp_2 = wp.zeros((N_GRID, N_GRID), dtype=wp.vec2f)
fft_temp_3 = wp.zeros((N_GRID, N_GRID), dtype=wp.vec2f)

Jog your memory about the three arrays that we allocated in the very beginning! 

| Array | Shape | Dtype | Purpose |
|-------|-------|-------|---------|
| `omega_0` | `(N_GRID, N_GRID)` | `float` | Vorticity at timestep $n$ |
| `omega_1` | `(N_GRID, N_GRID)` | `float` | Vorticity at timestep $n+1$ |
| `psi` | `(N_GRID, N_GRID)` | `float` | Stream function at any time |


### Exercise

- `inputs` in `wp.launch(viscous_advection_kernel, ...)` is missing one array that the kernel writes to.
- `wp.copy(dst, src)` -- between `omega_1` and `omega_0`, what should be the `dst` and what should be the `src`?

In [ ]:
def advance_vorticity_by_dt(omega_0, omega_1, psi):
    """Advance the vorticity field by one timestep.

    Orchestrates the finite-difference vorticity update (Building Block 1)
    and the FFT-based Poisson solve for the stream function (Building Block 2),
    then copies omega_1 back into omega_0 for the next iteration.

    Args:
        omega_0: Vorticity at timestep n (overwritten with omega_1 at exit).
        omega_1: Vorticity at timestep n+1 (written by this function).
        psi: Stream function (written by this function).
    """
    # ===== Building Block 1: Vorticity Update =====
    wp.launch(
        viscous_advection_kernel,
        dim=(N_GRID, N_GRID),
        inputs=[omega_0, psi, ...], # TODO: which array holds the output from the call here?
    )

    # ===== Building Block 2: Fourier Poisson Solver =====

    # Convert omega_1 (float32) to complex (vec2f)
    wp.launch(
        copy_float_to_complex,
        dim=(N_GRID, N_GRID),
        inputs=[omega_1, omega_complex],
    )

    # Forward 2D FFT: row FFT -> transpose -> row FFT
    wp.launch_tiled(
        fft_tiled,
        dim=[N_GRID, 1],
        inputs=[omega_complex, fft_temp_1],
        block_dim=N_GRID // 2,
    )
    wp.launch(
        transpose, dim=[N_GRID, N_GRID], inputs=[fft_temp_1, fft_temp_2],
    )
    wp.launch_tiled(
        fft_tiled,
        dim=[N_GRID, 1],
        inputs=[fft_temp_2, fft_temp_3],
        block_dim=N_GRID // 2,
    )

    # Solve in Fourier space: psi_hat = omega_hat / |k|^2
    wp.launch(
        multiply_k2_inverse,
        dim=(N_GRID, N_GRID),
        inputs=[inv_k_sq, fft_temp_3, fft_temp_1],
    )

    # Inverse 2D FFT: row IFFT -> transpose -> row IFFT
    wp.launch_tiled(
        ifft_tiled,
        dim=[N_GRID, 1],
        inputs=[fft_temp_1, fft_temp_2],
        block_dim=N_GRID // 2,
    )
    wp.launch(
        transpose, dim=[N_GRID, N_GRID], inputs=[fft_temp_2, fft_temp_3]
    )
    wp.launch_tiled(
        ifft_tiled,
        dim=[N_GRID, 1],
        inputs=[fft_temp_3, fft_temp_1],
        block_dim=N_GRID // 2,
    )

    # Extract real part and scale by 1/N^2
    wp.launch(
        extract_real_and_scale,
        dim=(N_GRID, N_GRID),
        inputs=[float(N_GRID * N_GRID), fft_temp_1, psi],
    )

    # Copy omega_1 back to omega_0 for next timestep
    wp.copy(..., ...) # TODO: what should be ``src`` and ``dest`` for ``wp.copy(dst, src)``

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
    # viscous_advection_kernel writes the updated vorticity into ``omega_1``
    inputs=[omega_0, psi, omega_1],
    # for the next timestep, ``omega_1`` (updated field) becomes the old field ``omega_0``
    wp.copy(omega_0, omega_1) 
```
</details>

### Full simulation

The solver is complete. Let's run the full simulation and visualize how the vorticity field evolves over time.

In [ ]:
from helpers.navier_stokes.utils import initialize_decaying_turbulence
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import os

# Re-initialize for a fresh run
omega_init_np = initialize_decaying_turbulence(n_grid=N_GRID, seed=42)
omega_0 = wp.array(omega_init_np, dtype=float)
omega_1 = wp.zeros_like(omega_0)
omega_hat = np.fft.fft2(omega_init_np)
psi_init_np = np.fft.ifft2(omega_hat * inv_k_sq_np).real.astype(np.float32)
psi = wp.array(psi_init_np, dtype=float)

NUM_FRAMES = 50
STEPS_PER_FRAME = 20
GIF_SIZE = 256

# Colormap setup
cmap = plt.cm.twilight
norm = Normalize(vmin=-15, vmax=15)

frames_baseline = []
print(f"Running {NUM_FRAMES} frames ({STEPS_PER_FRAME} steps between consecutive frames)...")

for frame in range(NUM_FRAMES):
    for _ in range(STEPS_PER_FRAME):
        advance_vorticity_by_dt(omega_0, omega_1, psi)

    # Capture frame (resize immediately to keep memory bounded)
    vorticity = omega_1.numpy().T  # Transpose for correct orientation
    colored = cmap(norm(vorticity))
    rgb = (colored[:, :, :3] * 255).astype(np.uint8)
    pil_frame = Image.fromarray(rgb).resize((GIF_SIZE, GIF_SIZE), Image.LANCZOS)
    frames_baseline.append(pil_frame)

    if (frame + 1) % 5 == 0:
        print(f"  Frame {frame + 1}/{NUM_FRAMES}")

total_steps = NUM_FRAMES * STEPS_PER_FRAME

In [ ]:
import IPython.display

os.makedirs("./images/navier-stokes", exist_ok=True)
output_file_baseline = f"./images/navier-stokes/turbulence_{N_GRID}x{N_GRID}_Re{int(RE)}.gif"

frames_baseline[0].save(
    output_file_baseline,
    save_all=True,
    append_images=frames_baseline[1:],
    duration=50,
    loop=0,
)

print(f"Saved: {output_file_baseline}")
IPython.display.Image(output_file_baseline)

---
## Using CUDA Graphs to Improve Simulation Performance

<p align="center">
  <img src="./images/navier-stokes/cuda_graph_timeline.svg" alt="CUDA graph timeline" width="800">
</p>

Our `advance_vorticity_by_dt()` function has a total of **10 kernel launches** that do relatively lightweight computations on a $512 \times 512$ grid. In this scenario, the overhead of launching kernels from the CPU side, queueing them on the GPU side, and other hidden costs can dominate the time the GPU spends doing the actual computations.

A neat trick is to use [CUDA graphs](https://pytorch.org/blog/accelerating-pytorch-with-cuda-graphs/). At a very broad level, **a CUDA graph allows us to define multiple kernel launches as a single unit**, rather than as a sequence of individually launched operations.

In this example, when we capture a single execution of ``advance_vorticity_by_dt()`` inside a CUDA graph, there is **only one CPU call to launch that graph and ultimately do all the computations on the GPU** instead of 10.

In Warp, ``wp.ScopedCapture()`` records all the CUDA operations, such as kernel launches, memory copies, and memory allocations, in a dependency graph. Thereafter, to actually execute the operations inside `advance_vorticity_by_dt()`, you simply replay the captured graph with `wp.capture_launch(graph)`

**Note**: While Warp supports memory allocations during graph capture, pre-allocating arrays with known sizes avoids allocation overhead on every replay and is considered best practice. 

```python
with wp.ScopedCapture() as capture:
    advance_vorticity_by_dt(...)           # record all operations into a graph
graph = capture.graph   # frozen execution plan

wp.capture_launch(graph)  # replay
```

Before incorporating a CUDA graph into our solver, let's see how long the original solver takes to run the simulation for 20,000 timesteps.

In [ ]:
import time

# Warmup: run for 10 steps
for _ in range(10):
    advance_vorticity_by_dt(omega_0, omega_1, psi)
wp.synchronize()

start = time.perf_counter()
for _ in range(20000):
    advance_vorticity_by_dt(omega_0, omega_1, psi)
wp.synchronize()
baseline_elapsed = time.perf_counter() - start
print(f"20000 steps: {baseline_elapsed:.3f}s ({20000 / baseline_elapsed:.0f} steps/s)")

Now, let's capture the `advance_vorticity_by_dt()` in a CUDA graph.

In [ ]:
from helpers.navier_stokes.utils import initialize_decaying_turbulence

# Re-initialize for a fresh run
omega_init_np = initialize_decaying_turbulence(n_grid=N_GRID, seed=42)
omega_0 = wp.array(omega_init_np, dtype=float)
omega_1 = wp.zeros_like(omega_0)
omega_hat = np.fft.fft2(omega_init_np)
psi_init_np = np.fft.ifft2(omega_hat * inv_k_sq_np).real.astype(np.float32)
psi = wp.array(psi_init_np, dtype=float)

# Capture advance_vorticity_by_dt() as a CUDA graph
with wp.ScopedCapture() as capture:
    advance_vorticity_by_dt(omega_0, omega_1, psi)
step_graph = capture.graph
print("CUDA graph captured successfully.")

Let's benchmark how long the same 20,000 steps take with a CUDA graph.

In [ ]:
# Compare: 20000 steps with CUDA graph
start = time.perf_counter()
for _ in range(20000):
    wp.capture_launch(step_graph)
wp.synchronize()
graph_elapsed = time.perf_counter() - start
print(f"20000 steps (graph):    {graph_elapsed:.3f}s ({20000 / graph_elapsed:.0f} steps/s)")
print(f"Speedup: {baseline_elapsed / graph_elapsed:.1f}x")

### Advantages of CUDA Graphs increase for compute-light kernels

<table align="center">
<tr>
<td align="center"><b>Without CUDA graph</b><br><img src="https://media.githubusercontent.com/media/NVIDIA/accelerated-computing-hub/main/tutorials/warp/notebooks/images/navier-stokes/nsys-no-graph.png" alt="Nsight Systems timeline without CUDA graph" width="600"></td>
<td align="center"><b>With CUDA graph</b><br><img src="https://media.githubusercontent.com/media/NVIDIA/accelerated-computing-hub/main/tutorials/warp/notebooks/images/navier-stokes/nsys-graph.png" alt="Nsight Systems timeline with CUDA graph" width="600"></td>
</tr>
</table>

These Nsight Systems profiles were collected at a resolution of 256x256. The advantage of CUDA graphs becomes more apparent as kernels become less compute-heavy, because launch and scheduling overhead can account for a larger fraction of the total step time when each kernel does relatively little work on the GPU.

Let's confirm our simulation results are unchanged when using the CUDA graph.

In [ ]:
# Re-initialize for a fresh run
omega_init_np = initialize_decaying_turbulence(n_grid=N_GRID, seed=42)
omega_0 = wp.array(omega_init_np, dtype=float)
omega_1 = wp.zeros_like(omega_0)
omega_hat = np.fft.fft2(omega_init_np)
psi_init_np = np.fft.ifft2(omega_hat * inv_k_sq_np).real.astype(np.float32)
psi = wp.array(psi_init_np, dtype=float)

# Capture the CUDA graph
with wp.ScopedCapture() as capture:
    advance_vorticity_by_dt(omega_0, omega_1, psi)
step_graph = capture.graph

NUM_FRAMES = 50
STEPS_PER_FRAME = 20
GIF_SIZE = 256

frames_graph = []
print(f"Running {NUM_FRAMES} frames ({STEPS_PER_FRAME} steps between consecutive frames)...")

start_time = time.perf_counter()
for frame in range(NUM_FRAMES):
    # Advance simulation
    for _ in range(STEPS_PER_FRAME):
        wp.capture_launch(step_graph)

    # Capture frame (resize immediately to keep memory bounded)
    vorticity = omega_1.numpy().T  # Transpose for correct orientation
    colored = cmap(norm(vorticity))
    rgb = (colored[:, :, :3] * 255).astype(np.uint8)
    pil_frame = Image.fromarray(rgb).resize((GIF_SIZE, GIF_SIZE), Image.LANCZOS)
    frames_graph.append(pil_frame)

    if (frame + 1) % 5 == 0:
        print(f"  Frame {frame + 1}/{NUM_FRAMES}")

elapsed = time.perf_counter() - start_time
total_steps = NUM_FRAMES * STEPS_PER_FRAME

In [ ]:
output_file_graph = f"./images/navier-stokes/turbulence_{N_GRID}x{N_GRID}_Re{int(RE)}_cuda_graph.gif"

frames_graph[0].save(
    output_file_graph,
    save_all=True,
    append_images=frames_graph[1:],
    duration=50,
    loop=0,
)

print(f"Saved: {output_file_graph}")
IPython.display.Image(output_file_graph)

---
## Conclusion

In this notebook, we used Warp's core features to build a complete physics solver. We learned:

- how to map a computational physics problem onto Warp's memory model,
- which operations can be implemented in a single kernel and which require coordination across multiple kernels,
- when CUDA Graphs may be worth exploring to reduce launch overhead and improve performance.

In the next notebook, we will look at the key steps required to make a solver like this differentiable.

---
## References

* 2-D Navier-Stokes solver example in Warp, https://github.com/NVIDIA/warp/blob/main/warp/examples/core/example_fft_poisson_navier_stokes_2d.py
* "Introducing Tile-Based Programming in Warp 1.5.0," NVIDIA Developer Blog, https://developer.nvidia.com/blog/introducing-tile-based-programming-in-warp-1-5-0/
* Documentation for tile programming in Warp, GitHub Pages, https://nvidia.github.io/warp/stable/user_guide/tiles.html
* "High-order methods for decaying two-dimensional homogeneous isotropic turbulence," DOI, https://doi.org/10.1016/j.compfluid.2012.04.026
* CUDA Graphs, NVIDIA Documentation, https://docs.nvidia.com/cuda/cuda-programming-guide/04-special-topics/cuda-graphs.html